Historical question: What spatial pattern is visible among the printing places represented in this teaching sample, and how do normalization, false imprints, uncertain attributions, and unmatched records affect that pattern?

The dataset contains 500 entries and is not representative. Place names are sometimes uncertain/inferred, this is noted in "place_relationship".

In [3]:
import pandas as pd

df = pd.read_csv("book-places-wikidata.csv")
coord_col = "coordinate location"

rows, cols = df.shape
print("Rows:", rows)
print("Columns:", cols)

print("\nColumn names:")
for name in df.columns:
    print(repr(name))

pd.set_option("display.max_columns", None)
print("\nFirst five rows:")
print(df.head(5))

print("\nUnique edition_id values:", df["edition_id"].nunique())
print("Total rows:", len(df))

check_cols = ["place", "place_interpreted", "wikidata_id", coord_col]
print("\nMissing values per column:")
print(df[check_cols].isna().sum())

distinct = df[coord_col].dropna().unique()
print("\nNumber of distinct coordinate values:", len(distinct))
print("\nFirst 20 distinct values:")
for value in distinct[:20]:
    print(repr(value))

Rows: 500
Columns: 19

Column names:
'edition_id'
'short_title'
'author'
'printer'
'place'
'place_norm'
'place_interpreted'
'coordinate location'
'country 2'
'wikidata_id'
'place_relationship'
'match_status'
'decision_note'
'year'
'language'
'format'
'country'
'year_numeric'
'decade'

First five rows:
   edition_id                                        short_title  \
0      627908                                 Das Newe Testament   
1      698366  Trewung vom liebebande oder hertztrewen. Item:...   
2      691468  Romischer Keyserlicher Majestat ordnung und Re...   
3      708446  Zwo predigten vom gantzen Christlichen wesen u...   
4      640867  Corpus juris civilis, institutiones dn. Joachi...   

                  author                 printer       place  place_norm  \
0                    NaN        Kunigunde Hergot    Nürnberg    Nürnberg   
1        Streuber, Peter       Zacharias Bärwald     Leipzig     Leipzig   
2                    NaN         Johann Schöffer       Mainz

In [5]:
# Coordinate format: one text cell "latitude,longitude" (two decimal numbers separated by a comma).
# Source order: latitude first, longitude second. The sample values match known city locations,
# e.g. Nuremberg "49.45...,11.07..." (about 49.5°N, 11.1°E) and Strasbourg "48.57...,7.75...".
# This is written for the dataframe `books`, which is copied, never changed.

import math
import pandas as pd

books = pd.read_csv("book-places-wikidata.csv")

coord_col = "coordinate location"
books_coordinates = books.copy()          # new dataframe; `books` stays untouched


def parse_coordinate(value):
    """Return (latitude, longitude, status) for one original coordinate cell."""
    if pd.isna(value) or str(value).strip() == "":
        return float("nan"), float("nan"), "missing"

    parts = str(value).split(",")
    if len(parts) != 2:
        return float("nan"), float("nan"), "format_error"

    try:
        lat = float(parts[0].strip())
        lon = float(parts[1].strip())
    except ValueError:
        return float("nan"), float("nan"), "format_error"

    if not (math.isfinite(lat) and math.isfinite(lon)):
        return float("nan"), float("nan"), "format_error"

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return float("nan"), float("nan"), "out_of_range"

    return lat, lon, "ok"


latitudes, longitudes, statuses = [], [], []
for value in books_coordinates[coord_col]:
    lat, lon, status = parse_coordinate(value)
    latitudes.append(lat)
    longitudes.append(lon)
    statuses.append(status)

books_coordinates["latitude"] = latitudes
books_coordinates["longitude"] = longitudes
books_coordinates["coordinate_parse_status"] = statuses

# --- Row counts ---
print("Input rows: ", len(books))
print("Output rows:", len(books_coordinates))
assert len(books) == len(books_coordinates), "Row count changed!"

# --- edition_id uniqueness ---
print("\nUnique edition_id values:", books_coordinates["edition_id"].nunique())
print("edition_id is unique:", books_coordinates["edition_id"].is_unique)
assert books_coordinates["edition_id"].is_unique, "Duplicate edition_id values found!"

# --- Usable coordinates ---
usable_mask = books_coordinates["coordinate_parse_status"] == "ok"
usable = int(usable_mask.sum())
print(f"\nRows with usable coordinates: {usable} of {len(books_coordinates)} "
      f"({usable / len(books_coordinates) * 100:.1f}%)")

# --- Unparseable original values ---
failed = books_coordinates.loc[~usable_mask, [coord_col, "coordinate_parse_status"]]
failed = failed.drop_duplicates()
print("\nDistinct original coordinate values that could not be parsed:")
if failed.empty:
    print("None")
else:
    print(failed.to_string(index=False))

# --- place_interpreted without any usable coordinates ---
has_usable = books_coordinates.groupby("place_interpreted")["coordinate_parse_status"].apply(
    lambda s: (s == "ok").any()
)
no_coords = sorted(has_usable[~has_usable].index.tolist())
print("\nDistinct place_interpreted values without usable coordinates:")
if no_coords:
    for place in no_coords:
        print(" -", place)
else:
    print("None")
print("Rows where place_interpreted itself is missing:",
      int(books_coordinates["place_interpreted"].isna().sum()))

# --- place_interpreted with more than one coordinate pair ---
pairs = (books_coordinates.loc[usable_mask, ["place_interpreted", "latitude", "longitude"]]
         .dropna(subset=["place_interpreted"])
         .drop_duplicates())

pairs_per_place = pairs.groupby("place_interpreted").size()
multi_coord_places = pairs_per_place[pairs_per_place > 1].index
print("\nplace_interpreted values with more than one coordinate pair:")
if len(multi_coord_places) == 0:
    print("None")
else:
    print(pairs[pairs["place_interpreted"].isin(multi_coord_places)]
          .sort_values("place_interpreted").to_string(index=False))

# --- Coordinate pairs shared by more than one place_interpreted ---
places_per_pair = pairs.groupby(["latitude", "longitude"]).size()
multi_place_pairs = places_per_pair[places_per_pair > 1].index
print("\nCoordinate pairs assigned to more than one place_interpreted value:")
if len(multi_place_pairs) == 0:
    print("None")
else:
    shared = pairs.set_index(["latitude", "longitude"]).loc[multi_place_pairs].reset_index()
    print(shared.sort_values(["latitude", "longitude"]).to_string(index=False))

# --- Counts by match_status and coordinate_parse_status ---
print("\nCounts by match_status and coordinate_parse_status:")
match_status_filled = books_coordinates["match_status"].fillna("(missing)")
print(pd.crosstab(match_status_filled, books_coordinates["coordinate_parse_status"], margins=True))

# --- Representative rows (up to 3 per parse status) ---
show_cols = ["place", "place_interpreted", "match_status", coord_col,
             "latitude", "longitude", "coordinate_parse_status"]
print("\nRepresentative rows (up to 3 per coordinate_parse_status):")
sample_rows = (books_coordinates
               .sort_values("coordinate_parse_status")
               .groupby("coordinate_parse_status")
               .head(3)[show_cols])
display(sample_rows)

Input rows:  500
Output rows: 500

Unique edition_id values: 500
edition_id is unique: True

Rows with usable coordinates: 499 of 500 (99.8%)

Distinct original coordinate values that could not be parsed:
coordinate location coordinate_parse_status
                NaN                 missing

Distinct place_interpreted values without usable coordinates:
 - Homberg
Rows where place_interpreted itself is missing: 0

place_interpreted values with more than one coordinate pair:
None

Coordinate pairs assigned to more than one place_interpreted value:
None

Counts by match_status and coordinate_parse_status:
coordinate_parse_status  missing   ok  All
match_status                              
confirmed                      0  499  499
not_geocoded                   1    0    1
All                            1  499  500

Representative rows (up to 3 per coordinate_parse_status):


,place,place_interpreted,match_status,coordinate location,latitude,longitude,coordinate_parse_status
174,Erfurt (=Homberg),Homberg,not_geocoded,NaN,NaN,NaN,missing
0,Nürnberg,Nuremberg,confirmed,"49.45388888888889,11.0775",49.453889,11.077500,ok
340,Köln,Cologne,confirmed,"50.942222222222,6.9577777777778",50.942222,6.957778,ok
339,Basel,Basel,confirmed,"47.558055555555555,7.587777777777777",47.558056,7.587778,ok


In [6]:
import pandas as pd

# books_coordinates is only read here; it is never changed.

# --- edition_id check on the original dataframe ---
print("Rows in books_coordinates:", len(books_coordinates))
print("Unique edition_id values:", books_coordinates["edition_id"].nunique())
print("edition_id is unique:", books_coordinates["edition_id"].is_unique)
assert books_coordinates["edition_id"].is_unique, "Duplicate edition_id values found!"

# --- Split into usable and excluded rows ---
usable_mask = books_coordinates["latitude"].notna() & books_coordinates["longitude"].notna()
usable_rows = books_coordinates[usable_mask]
excluded_rows = books_coordinates[~usable_mask]

print("\nRecords with usable coordinates:", len(usable_rows))
print("Records excluded (missing or invalid coordinates):", len(excluded_rows))

print("\nDistinct place_interpreted and match_status values among excluded records:")
if excluded_rows.empty:
    print("None")
else:
    excluded_summary = (excluded_rows[["place_interpreted", "match_status"]]
                        .fillna("(missing)")
                        .drop_duplicates()
                        .sort_values(["place_interpreted", "match_status"]))
    print(excluded_summary.to_string(index=False))

# --- One place_interpreted with more than one coordinate pair? ---
pairs = (usable_rows[["place_interpreted", "latitude", "longitude"]]
         .dropna(subset=["place_interpreted"])
         .drop_duplicates())
pairs_per_place = pairs.groupby("place_interpreted").size()
multi_coord_places = pairs_per_place[pairs_per_place > 1].index

print("\nplace_interpreted values with more than one coordinate pair (for review):")
if len(multi_coord_places) == 0:
    print("None")
else:
    print(pairs[pairs["place_interpreted"].isin(multi_coord_places)]
          .sort_values("place_interpreted").to_string(index=False))

# --- Aggregate: one row per place_interpreted + latitude + longitude ---
city_counts = (usable_rows
               .groupby(["place_interpreted", "latitude", "longitude"], dropna=False)
               .agg(editions=("edition_id", "count"))
               .reset_index()
               .sort_values("editions", ascending=False)
               .reset_index(drop=True))

print("\ncity_counts (largest to smallest):")
print(city_counts.to_string())

# --- Totals must match ---
print("\nRows with usable coordinates:", len(usable_rows))
print("Sum of editions in city_counts:", city_counts["editions"].sum())
assert len(usable_rows) == city_counts["editions"].sum(), "Totals do not match!"

# --- Save without the pandas index ---
city_counts.to_csv("printing_places_aggregated.csv", index=False, encoding="utf-8")
print("\nSaved printing_places_aggregated.csv with", len(city_counts), "rows")

Rows in books_coordinates: 500
Unique edition_id values: 500
edition_id is unique: True

Records with usable coordinates: 499
Records excluded (missing or invalid coordinates): 1

Distinct place_interpreted and match_status values among excluded records:
place_interpreted match_status
          Homberg not_geocoded

place_interpreted values with more than one coordinate pair (for review):
None

city_counts (largest to smallest):
     place_interpreted   latitude  longitude  editions
0                Basel  47.558056   7.587778       136
1              Cologne  50.942222   6.957778        69
2            Nuremberg  49.453889  11.077500        69
3           Strasbourg  48.573333   7.752222        56
4              Leipzig  51.340632  12.374733        49
5             Augsburg  48.368889  10.897778        39
6               Zurich  47.374444   8.541111        18
7               Erfurt  50.978056  11.028889        15
8             Tübingen  48.520000   9.055556        14
9            Magd

In [8]:
import html
import math

import folium

# city_counts is only read here; it is never changed.

# Map with no default tiles, centered on Central Europe (zoom 5 or lower).
printing_map = folium.Map(
    location=[50.0, 10.0],
    zoom_start=4,
    max_zoom=6,
    tiles=None,
)

# Custom low-resolution basemap.
folium.TileLayer(
    tiles="https://tiles.openfreemap.org/natural_earth/ne2sr/{z}/{x}/{y}.png",
    name="OpenFreeMap Natural Earth",
    attr=(
        '&copy; <a href="https://openfreemap.org">OpenFreeMap</a> '
        '&amp; <a href="https://www.naturalearthdata.com">Natural Earth</a>'
    ),
    max_zoom=6,
).add_to(printing_map)

# One circle marker per row.
points = []
for row in city_counts.itertuples(index=False):
    place = row.place_interpreted if isinstance(row.place_interpreted, str) else "(no place name)"
    place = html.escape(place)
    editions = int(row.editions)
    lat = float(row.latitude)
    lon = float(row.longitude)

    radius = 3 + 2.2 * math.sqrt(editions)

    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color="#8b1e3f",
        weight=1,
        fill=True,
        fill_color="#c0392b",
        fill_opacity=0.6,
        tooltip=f"{place}: {editions} edition(s)",
        popup=folium.Popup(
            f"<b>{place}</b><br>"
            f"Editions: {editions}<br>"
            f"Latitude: {lat:.5f}<br>"
            f"Longitude: {lon:.5f}",
            max_width=250,
        ),
    ).add_to(printing_map)

    points.append([lat, lon])

# Fit the view to all points, without zooming in past level 5.
if points:
    printing_map.fit_bounds(points, padding=(30, 30), max_zoom=5)

folium.LayerControl().add_to(printing_map)

# Save and display.
printing_map.save("printing_places_map.html")
printing_map

In [9]:
#!/usr/bin/env python3
"""
Build an interactive "Printing Places Explorer" web page from the OpenRefine CSV.

Usage (in a terminal, or in Jupyter with a leading "!"):
    python build_printing_places_site.py
    python build_printing_places_site.py my_input.csv my_output.html

What it does
    1. Reads the CSV WITHOUT changing it (every cell is read as plain text).
    2. Parses the "coordinate location" column (format "latitude,longitude").
    3. Embeds the data in ONE self-contained HTML file (printing_places_explorer.html).
    4. The page uses Leaflet (map) and Chart.js (charts), loaded from the internet.

The original CSV is never modified or overwritten.
"""

import json
import math
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

# ----------------------------------------------------------------------------
# Settings you may want to change
# ----------------------------------------------------------------------------
INPUT_CSV = sys.argv[1] if len(sys.argv) > 1 else "book-places-wikidata.csv"
OUTPUT_HTML = sys.argv[2] if len(sys.argv) > 2 else "printing_places_dashboard.html"

COORD_COL = "coordinate location"   # format: "latitude,longitude"

# Some OpenRefine exports show text like "NÃ¼rnberg" instead of "Nürnberg".
# Set this to True to repair such text ONLY in the web page (the CSV stays as it is).
FIX_MOJIBAKE = False


# ----------------------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------------------
def fix_text(value):
    """Try to repair text that was UTF-8 but was read as Latin-1 (e.g. 'NÃ¼rnberg')."""
    if not FIX_MOJIBAKE:
        return value
    try:
        return value.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value


def parse_coordinate(value):
    """Return (latitude, longitude, status) for one original coordinate cell.

    Observed format: "latitude,longitude", for example "49.45388888888889,11.0775".
    """
    if value is None or str(value).strip() == "":
        return None, None, "missing"

    parts = str(value).split(",")
    if len(parts) != 2:
        return None, None, "format_error"

    try:
        lat = float(parts[0].strip())
        lon = float(parts[1].strip())
    except ValueError:
        return None, None, "format_error"

    if not (math.isfinite(lat) and math.isfinite(lon)):
        return None, None, "format_error"

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return None, None, "out_of_range"

    return lat, lon, "ok"


def get_year(row):
    """Read a numeric year from 'year_numeric', or else from 'year'. None if not possible."""
    for name in ("year_numeric", "year"):
        text = str(row.get(name, "")).strip()
        if not text:
            continue
        try:
            return int(float(text))
        except ValueError:
            match = re.search(r"\d{4}", text)
            if match:
                return int(match.group())
    return None


def label(value):
    """Use '(blank)' for empty values so they can be filtered and charted."""
    value = str(value).strip()
    return value if value else "(blank)"


# ----------------------------------------------------------------------------
# 1. Load the CSV (read-only, everything as text, nothing dropped)
# ----------------------------------------------------------------------------
books = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
print(f"Loaded {len(books)} rows and {len(books.columns)} columns from {INPUT_CSV}")

for needed in ("edition_id", "place_interpreted", COORD_COL):
    if needed not in books.columns:
        sys.exit(f"ERROR: the column '{needed}' was not found. Columns are: {list(books.columns)}")

if not books["edition_id"].is_unique:
    print("WARNING: edition_id is not unique. Please review the duplicates.")
else:
    print("edition_id is unique:", books["edition_id"].nunique(), "values")

columns = list(books.columns)

# ----------------------------------------------------------------------------
# 2. Build one record per row
# ----------------------------------------------------------------------------
records = []
for i, row in enumerate(books.to_dict("records")):
    cells = [fix_text(str(row[c])) for c in columns]
    clean = {c: fix_text(str(row[c])).strip() for c in columns}
    lat, lon, status = parse_coordinate(row[COORD_COL])
    records.append({
        "i": i,
        "id": clean.get("edition_id", ""),
        "place": clean.get("place", ""),
        "pi": clean.get("place_interpreted", ""),
        "rel": label(clean.get("place_relationship", "")),
        "ms": label(clean.get("match_status", "")),
        "lang": label(clean.get("language", "")),
        "fmt": label(clean.get("format", "")),
        "printer": clean.get("printer", ""),
        "year": get_year(clean),
        "lat": lat,
        "lon": lon,
        "st": status,
        "cells": cells,
    })

assert len(records) == len(books), "Row count changed!"

# ----------------------------------------------------------------------------
# 3. Summary information for the page
# ----------------------------------------------------------------------------
rel_counts = Counter(r["rel"] for r in records)
relationships = [k for k, _ in rel_counts.most_common()]
languages = sorted({r["lang"] for r in records})
formats = sorted({r["fmt"] for r in records})
match_statuses = sorted({r["ms"] for r in records})

# Data-quality review (same checks as in the earlier validation steps)
status_counts = dict(Counter(r["st"] for r in records))
pairs_per_place = defaultdict(set)
places_per_pair = defaultdict(set)
for r in records:
    if r["lat"] is not None and r["pi"]:
        pairs_per_place[r["pi"]].add((r["lat"], r["lon"]))
        places_per_pair[(r["lat"], r["lon"])].add(r["pi"])

quality = {
    "status_counts": status_counts,
    "multi_coord_places": [
        {"place": p, "pairs": sorted(list(v))}
        for p, v in sorted(pairs_per_place.items()) if len(v) > 1
    ],
    "shared_pairs": [
        {"lat": k[0], "lon": k[1], "places": sorted(v)}
        for k, v in sorted(places_per_pair.items()) if len(v) > 1
    ],
}

payload = {
    "columns": columns,
    "records": records,
    "relationships": relationships,
    "languages": languages,
    "formats": formats,
    "match_statuses": match_statuses,
    "quality": quality,
}
payload_json = json.dumps(payload, ensure_ascii=False).replace("</", "<\\/")

# ----------------------------------------------------------------------------
# 4. The web page (HTML + CSS + JavaScript)
# ----------------------------------------------------------------------------
HTML = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Printing Places Explorer</title>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css">
<style>
:root{--bg:#f5f3ee;--card:#fff;--ink:#222;--muted:#6b6b6b;--accent:#8b1e3f;--line:#ddd6c8;}
*{box-sizing:border-box}
body{margin:0;font-family:system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;background:var(--bg);color:var(--ink);}
header{padding:18px 24px 4px}
h1{margin:0;font-size:1.5rem}
#subtitle{margin:4px 0 0;color:var(--muted);font-size:.9rem}
.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:12px 14px}
.filters{display:flex;flex-wrap:wrap;gap:10px 16px;align-items:flex-end;margin:12px 24px}
.filters label{display:flex;flex-direction:column;font-size:.75rem;color:var(--muted);gap:3px}
.filters input[type=search],.filters select{padding:6px 8px;border:1px solid var(--line);border-radius:6px;font-size:.85rem;min-width:140px}
.relbox{display:flex;flex-wrap:wrap;gap:4px 12px;font-size:.8rem;color:var(--ink)}
.relbox label{flex-direction:row;align-items:center;gap:4px;color:var(--ink);font-size:.8rem}
.swatch{display:inline-block;width:11px;height:11px;border-radius:50%;margin-right:5px;vertical-align:-1px}
button{padding:6px 12px;border:1px solid var(--line);background:#fff;border-radius:6px;cursor:pointer;font-size:.85rem}
button:hover{background:#faf6ea}
.chip{display:inline-block;background:#fde9ee;border:1px solid #e6b3c1;border-radius:14px;padding:3px 10px;font-size:.8rem;margin-left:6px;cursor:pointer}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:0 24px 14px}
.kpi b{display:block;font-size:1.4rem}
.kpi span{color:var(--muted);font-size:.78rem}
.main{display:grid;grid-template-columns:minmax(0,3fr) minmax(0,2fr);gap:14px;margin:0 24px 14px}
@media(max-width:1000px){.main{grid-template-columns:1fr}}
#map{height:560px;border-radius:10px;border:1px solid var(--line);background:#dfe9f0}
.timebar{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-top:10px;font-size:.85rem}
.timebar input[type=range]{flex:1;min-width:160px}
.trail{margin-top:10px;font-size:.85rem}
.trail input{padding:6px 8px;border:1px solid var(--line);border-radius:6px;min-width:240px}
.trail ol{margin:6px 0 0 18px;padding:0}
.charts{display:grid;grid-template-columns:1fr 1fr;gap:12px;align-content:start}
.chartbox{height:230px;position:relative}
.span2{grid-column:1/-1}
.legend{background:rgba(255,255,255,.96);padding:8px 10px;border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.25);font-size:12px;max-width:250px;line-height:1.55}
.trail-num{background:#1d4e89;color:#fff;border-radius:50%;width:20px;height:20px;text-align:center;line-height:16px;font-size:11px;font-weight:600;border:2px solid #fff}
.pie-icon{background:none;border:none;filter:drop-shadow(0 1px 3px rgba(0,0,0,.45))}
.tablecard{margin:0 24px 30px}
.tablebar{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-bottom:8px;font-size:.85rem}
.tablewrap{overflow:auto;max-height:520px;border:1px solid var(--line);border-radius:6px}
table{border-collapse:collapse;width:max-content;min-width:100%;font-size:.8rem}
th{position:sticky;top:0;background:#efe9db;cursor:pointer;white-space:nowrap;text-align:left;padding:6px 8px;z-index:1}
td{padding:5px 8px;border-top:1px solid #eee;max-width:280px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
tr:hover td{background:#faf6ea}
tr.sel td{background:#fde9ee}
.plink{color:#1d4e89;text-decoration:underline;cursor:pointer}
details{font-size:.82rem}
details ul{margin:4px 0 8px 18px;padding:0}
</style>
</head>
<body>
<header>
  <h1>Printing Places Explorer</h1>
  <p id="subtitle"></p>
</header>

<section class="card filters" id="filters">
  <label>Search all columns<input type="search" id="q" placeholder="title, printer, place…"></label>
  <label>Language<select id="fLang"></select></label>
  <label>Format<select id="fFmt"></select></label>
  <label>Match status<select id="fMs"></select></label>
  <label>Years: <span id="yLabel"></span>
    <span><input type="range" id="yMin"> <input type="range" id="yMax"></span></label>
  <label>Place relationship<span class="relbox" id="relBoxes"></span></label>
  <span><button id="resetBtn">Reset filters</button> <span id="chips"></span></span>
</section>

<section class="kpis" id="kpis"></section>

<section class="main">
  <div>
    <div id="map"></div>
    <div class="card timebar">
      <b>Time Machine</b>
      <select id="timeMode">
        <option value="off">Off</option>
        <option value="cum">Cumulative (up to a decade)</option>
        <option value="win">One decade at a time</option>
      </select>
      <button id="playBtn">▶ Play</button>
      <input type="range" id="timeSlider" step="10" disabled>
      <span id="timeLabel"></span>
    </div>
    <div class="card trail">
      <b>Printer Trails</b> – follow a printer from place to place, in date order<br>
      <input id="printerInput" list="printerList" placeholder="Type or pick a printer…">
      <datalist id="printerList"></datalist>
      <button id="clearTrail">Clear trail</button>
      <div id="trailInfo" style="margin-top:6px;color:#6b6b6b">Only printers recorded in two or more places are listed. You can also click a printer name in the table.</div>
    </div>
  </div>
  <div class="charts">
    <div class="card chartbox span2"><canvas id="cDecade"></canvas></div>
    <div class="card chartbox"><canvas id="cRel"></canvas></div>
    <div class="card chartbox"><canvas id="cLang"></canvas></div>
    <div class="card chartbox span2"><canvas id="cPlaces"></canvas></div>
    <details class="card span2"><summary><b>Data quality review</b></summary><div id="quality"></div></details>
  </div>
</section>

<section class="card tablecard">
  <div class="tablebar">
    <b>Books</b>
    <span id="tableInfo"></span>
    <button id="zoomBtn">Zoom map to these records</button>
    <span style="flex:1"></span>
    <label>Rows per page <select id="pageSize"><option>25</option><option>50</option><option>100</option></select></label>
    <button id="prevPg">‹ Prev</button><span id="pgInfo"></span><button id="nextPg">Next ›</button>
  </div>
  <div class="tablewrap"><table><thead id="thead"></thead><tbody id="tbody"></tbody></table></div>
  <p style="color:#6b6b6b;font-size:.78rem;margin:6px 0 0">Click a row to select its place on the map. Click a column heading to sort. The last column is derived from the coordinate parsing; all other columns are the original CSV.</p>
</section>

<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<script>
(function(){
const DATA = __DATA__;
const R = DATA.records;
const COLS = DATA.columns;
const rels = DATA.relationships;
const PALETTE = ["#1f77b4","#e8743b","#2ca02c","#9467bd","#d4a017","#17a2b8","#d62728","#7f7f7f","#8c564b","#e377c2"];
const relColor = {};
rels.forEach((v,i)=>{ relColor[v] = PALETTE[i % PALETTE.length]; });

const $ = id => document.getElementById(id);
const esc = s => String(s).replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));

R.forEach(r => {
  r.hay = r.cells.join(" ").toLowerCase();
  r.key = r.lat == null ? null : (r.pi || "(no place name)") + "|" + r.lat + "|" + r.lon;
});
const years = R.map(r => r.year).filter(y => y != null);
const Y0 = years.length ? Math.min(...years) : 1500;
const Y1 = years.length ? Math.max(...years) : 1700;
const D0 = Math.floor(Y0/10)*10, D1 = Math.floor(Y1/10)*10;
const PRINTER_COL = COLS.indexOf("printer");

const S = { q:"", lang:"", fmt:"", ms:"", rels:new Set(rels), yMin:Y0, yMax:Y1,
            place:null, mode:"off", t:D0, sortCol:null, sortDir:1, page:0,
            groups:new Map(), scope:[], filtered:[] };

// ---------------------------------------------------------------- subtitle
const usableAll = R.filter(r => r.lat != null).length;
$("subtitle").textContent = R.length + " records · " + usableAll + " with usable coordinates (" +
  (R.length ? (usableAll/R.length*100).toFixed(1) : 0) + "%) · years " + Y0 + "–" + Y1;

// ---------------------------------------------------------------- controls setup
function fillSelect(id, values){
  $(id).innerHTML = '<option value="">All</option>' + values.map(v => '<option>'+esc(v)+'</option>').join("");
}
fillSelect("fLang", DATA.languages);
fillSelect("fFmt", DATA.formats);
fillSelect("fMs", DATA.match_statuses);
["yMin","yMax"].forEach(id => { $(id).min = Y0; $(id).max = Y1; });
$("yMin").value = Y0; $("yMax").value = Y1;
$("timeSlider").min = D0; $("timeSlider").max = D1; $("timeSlider").value = D0;

$("relBoxes").innerHTML = rels.map((k,i) =>
  '<label><input type="checkbox" data-rel="'+esc(k)+'" checked><span class="swatch" style="background:'+relColor[k]+'"></span>'+esc(k)+'</label>').join("");

$("q").addEventListener("input", () => { S.q = $("q").value; S.page = 0; render(); });
$("fLang").addEventListener("change", () => { S.lang = $("fLang").value; S.page = 0; render(); });
$("fFmt").addEventListener("change", () => { S.fmt = $("fFmt").value; S.page = 0; render(); });
$("fMs").addEventListener("change", () => { S.ms = $("fMs").value; S.page = 0; render(); });
$("relBoxes").addEventListener("change", e => {
  const k = e.target.getAttribute("data-rel");
  if (e.target.checked) S.rels.add(k); else S.rels.delete(k);
  S.page = 0; render();
});
$("yMin").addEventListener("input", () => {
  S.yMin = +$("yMin").value;
  if (S.yMin > S.yMax) { S.yMax = S.yMin; $("yMax").value = S.yMax; }
  S.page = 0; render();
});
$("yMax").addEventListener("input", () => {
  S.yMax = +$("yMax").value;
  if (S.yMax < S.yMin) { S.yMin = S.yMax; $("yMin").value = S.yMin; }
  S.page = 0; render();
});
$("resetBtn").addEventListener("click", () => {
  stopPlay();
  S.q = ""; S.lang = ""; S.fmt = ""; S.ms = ""; S.rels = new Set(rels);
  S.yMin = Y0; S.yMax = Y1; S.place = null; S.mode = "off"; S.t = D0; S.page = 0;
  $("q").value = ""; $("fLang").value = ""; $("fFmt").value = ""; $("fMs").value = "";
  $("yMin").value = Y0; $("yMax").value = Y1; $("timeMode").value = "off"; $("timeSlider").value = D0;
  document.querySelectorAll("#relBoxes input").forEach(b => b.checked = true);
  render();
});

// ---------------------------------------------------------------- filtering
function currentRange(){
  if (S.mode === "cum") return [Y0, S.t + 9];
  if (S.mode === "win") return [S.t, S.t + 9];
  return [S.yMin, S.yMax];
}
function fullRange(){ return S.mode === "off" && S.yMin === Y0 && S.yMax === Y1; }

function filterRows(){
  const [lo, hi] = currentRange();
  const full = fullRange();
  const q = S.q.trim().toLowerCase();
  return R.filter(r => {
    if (q && !r.hay.includes(q)) return false;
    if (S.lang && r.lang !== S.lang) return false;
    if (S.fmt && r.fmt !== S.fmt) return false;
    if (S.ms && r.ms !== S.ms) return false;
    if (!S.rels.has(r.rel)) return false;
    if (r.year == null) { if (!full) return false; }
    else if (r.year < lo || r.year > hi) return false;
    return true;
  });
}

function groupPlaces(rows){
  const m = new Map();
  rows.forEach(r => {
    if (!r.key) return;
    let g = m.get(r.key);
    if (!g) { g = { key:r.key, name:r.pi || "(no place name)", lat:r.lat, lon:r.lon, n:0, rc:{} }; m.set(r.key, g); }
    g.n++;
    g.rc[r.rel] = (g.rc[r.rel] || 0) + 1;
  });
  return m;
}

// ---------------------------------------------------------------- map
const map = L.map("map", { center:[50,10], zoom:4, minZoom:3, maxZoom:6 });
const tiles = L.tileLayer("https://tiles.openfreemap.org/natural_earth/ne2sr/{z}/{x}/{y}.png", {
  maxZoom: 6, maxNativeZoom: 6,
  attribution: '&copy; <a href="https://openfreemap.org">OpenFreeMap</a> &amp; <a href="https://www.naturalearthdata.com">Natural Earth</a>'
}).addTo(map);
const circles = L.layerGroup().addTo(map);
const trailLayer = L.layerGroup().addTo(map);
L.control.layers({ "OpenFreeMap Natural Earth": tiles }, { "Printer trail": trailLayer }, { collapsed:true }).addTo(map);

const legend = L.control({ position: "bottomleft" });
legend.onAdd = function(){
  const div = L.DomUtil.create("div", "legend");
  L.DomEvent.disableClickPropagation(div);
  return div;
};
legend.addTo(map);

map.on("click", () => { if (S.place) { S.place = null; render(); } });

function pieSVG(g, d){
  const c = d/2, r = c - 2, total = g.n;
  const ents = rels.filter(k => g.rc[k]).map(k => [k, g.rc[k]]);
  let out = '<svg width="'+d+'" height="'+d+'" viewBox="0 0 '+d+' '+d+'" xmlns="http://www.w3.org/2000/svg">';
  if (ents.length === 1) {
    out += '<circle cx="'+c+'" cy="'+c+'" r="'+r+'" fill="'+relColor[ents[0][0]]+'" stroke="#fff" stroke-width="2"><title>'+esc(ents[0][0])+': '+ents[0][1]+'</title></circle>';
  } else {
    let a0 = -Math.PI/2;
    ents.forEach(([k, n]) => {
      const a1 = a0 + 2*Math.PI*n/total;
      const x0 = c + r*Math.cos(a0), y0 = c + r*Math.sin(a0);
      const x1 = c + r*Math.cos(a1), y1 = c + r*Math.sin(a1);
      const large = (a1 - a0) > Math.PI ? 1 : 0;
      out += '<path d="M'+c+' '+c+' L'+x0+' '+y0+' A'+r+' '+r+' 0 '+large+' 1 '+x1+' '+y1+' Z" fill="'+relColor[k]+'" stroke="#fff" stroke-width="2"><title>'+esc(k)+': '+n+'</title></path>';
      a0 = a1;
    });
  }
  out += '<circle cx="'+c+'" cy="'+c+'" r="'+(r*0.42)+'" fill="#fff"/><text x="'+c+'" y="'+(c+4)+'" text-anchor="middle" font-size="12" font-weight="700" fill="#222">'+total+'</text></svg>';
  return out;
}

function drawMap(){
  circles.clearLayers();
  S.groups.forEach(g => {
    const rad = 3 + 2.2*Math.sqrt(g.n);
    const tip = esc(g.name) + ": " + g.n + " edition" + (g.n === 1 ? "" : "s");
    if (g.key === S.place) {
      const d = Math.max(64, 2*rad + 18);
      const icon = L.divIcon({ className:"pie-icon", html:pieSVG(g, d), iconSize:[d,d], iconAnchor:[d/2,d/2] });
      L.marker([g.lat, g.lon], { icon:icon, zIndexOffset:1000 })
        .bindTooltip(tip)
        .on("click", e => { L.DomEvent.stopPropagation(e); S.place = null; render(); })
        .addTo(circles);
    } else {
      L.circleMarker([g.lat, g.lon], { radius:rad, color:"#6d1230", weight:1, fillColor:"#c0392b", fillOpacity:0.6 })
        .bindTooltip(tip)
        .on("click", e => { L.DomEvent.stopPropagation(e); selectPlace(g.key, false); })
        .addTo(circles);
    }
  });
  trailLayer.eachLayer(l => { if (l.bringToFront) l.bringToFront(); });
}

function updateLegend(filtered){
  const el = legend.getContainer();
  const sw = k => '<span class="swatch" style="background:'+relColor[k]+'"></span>';
  if (S.place && S.groups.has(S.place)) {
    const g = S.groups.get(S.place);
    el.innerHTML = '<b>'+esc(g.name)+'</b><br>'+g.n+' edition'+(g.n===1?'':'s')+
      '<br><small>lat '+g.lat.toFixed(4)+', lon '+g.lon.toFixed(4)+'</small><hr style="margin:4px 0">'+
      rels.filter(k => g.rc[k]).map(k => sw(k)+esc(k)+': <b>'+g.rc[k]+'</b> ('+Math.round(g.rc[k]/g.n*100)+'%)').join("<br>")+
      '<br><small style="color:#6b6b6b">Click the pie or the map to close.</small>';
  } else {
    const counts = {};
    filtered.forEach(r => { counts[r.rel] = (counts[r.rel] || 0) + 1; });
    el.innerHTML = '<b>Place relationships</b><br><small style="color:#6b6b6b">Click a circle to turn it into a pie chart.</small><hr style="margin:4px 0">'+
      rels.map(k => sw(k)+esc(k)+': '+(counts[k] || 0)).join("<br>");
  }
}

function selectPlace(key, fly){
  S.place = key;
  S.page = 0;
  render();
  if (fly && S.groups.has(key)) {
    const g = S.groups.get(key);
    map.flyTo([g.lat, g.lon], Math.max(5, Math.min(6, map.getZoom())), { duration:0.6 });
  }
}

function fitToRows(rows){
  const pts = rows.filter(r => r.lat != null).map(r => [r.lat, r.lon]);
  if (pts.length) map.fitBounds(pts, { padding:[30,30], maxZoom:5 });
}

// ---------------------------------------------------------------- KPIs
function renderKPIs(scope){
  const mapped = scope.filter(r => r.lat != null);
  const places = new Set(mapped.map(r => r.key)).size;
  const printers = new Set(scope.map(r => r.printer).filter(Boolean)).size;
  const ys = scope.map(r => r.year).filter(y => y != null);
  const yr = ys.length ? Math.min(...ys) + "–" + Math.max(...ys) : "–";
  const cov = scope.length ? (mapped.length/scope.length*100).toFixed(0) + "%" : "–";
  const card = (v, t) => '<div class="card kpi"><b>'+v+'</b><span>'+t+'</span></div>';
  $("kpis").innerHTML = card(scope.length + " / " + R.length, "records shown") +
    card(places, "places on the map") + card(cov, "records with coordinates") +
    card(printers, "distinct printers") + card(yr, "years covered");
}

// ---------------------------------------------------------------- charts
Chart.defaults.font.size = 11;
const charts = {};
const title = t => ({ display:true, text:t });
charts.decade = new Chart($("cDecade"), { type:"bar", data:{ labels:[], datasets:[] },
  options:{ responsive:true, maintainAspectRatio:false, animation:{ duration:200 },
    scales:{ x:{ stacked:true }, y:{ stacked:true, beginAtZero:true } },
    plugins:{ legend:{ display:false }, title:title("Editions by decade (colour = place relationship)") } } });
charts.rel = new Chart($("cRel"), { type:"doughnut", data:{ labels:[], datasets:[{ data:[], backgroundColor:[] }] },
  options:{ responsive:true, maintainAspectRatio:false, animation:{ duration:200 },
    plugins:{ legend:{ position:"right", labels:{ boxWidth:10 } }, title:title("Place relationships") } } });
charts.lang = new Chart($("cLang"), { type:"bar", data:{ labels:[], datasets:[{ data:[], backgroundColor:"#8b1e3f" }] },
  options:{ responsive:true, maintainAspectRatio:false, animation:{ duration:200 },
    plugins:{ legend:{ display:false }, title:title("Languages") }, scales:{ y:{ beginAtZero:true } } } });
charts.places = new Chart($("cPlaces"), { type:"bar", data:{ labels:[], datasets:[{ data:[], backgroundColor:"#c0392b" }] },
  options:{ indexAxis:"y", responsive:true, maintainAspectRatio:false, animation:{ duration:200 },
    plugins:{ legend:{ display:false }, title:title("Top 10 places by number of editions") }, scales:{ x:{ beginAtZero:true } } } });

function tally(rows, fn){
  const m = {};
  rows.forEach(r => { const k = fn(r); if (k) m[k] = (m[k] || 0) + 1; });
  return Object.entries(m).sort((a,b) => b[1] - a[1]);
}

function renderCharts(scope){
  const decades = [];
  for (let d = D0; d <= D1; d += 10) decades.push(d);
  const grid = {};
  rels.forEach(k => { grid[k] = {}; });
  scope.forEach(r => {
    if (r.year == null) return;
    const d = Math.floor(r.year/10)*10;
    grid[r.rel][d] = (grid[r.rel][d] || 0) + 1;
  });
  charts.decade.data.labels = decades.map(d => d + "s");
  charts.decade.data.datasets = rels.map(k => ({ label:k, backgroundColor:relColor[k], data:decades.map(d => grid[k][d] || 0) }));
  charts.decade.update();

  const rt = tally(scope, r => r.rel);
  charts.rel.data.labels = rt.map(x => x[0]);
  charts.rel.data.datasets[0].data = rt.map(x => x[1]);
  charts.rel.data.datasets[0].backgroundColor = rt.map(x => relColor[x[0]]);
  charts.rel.update();

  const lt = tally(scope, r => r.lang).slice(0, 8);
  charts.lang.data.labels = lt.map(x => x[0]);
  charts.lang.data.datasets[0].data = lt.map(x => x[1]);
  charts.lang.update();

  const pt = tally(scope, r => r.pi).slice(0, 10);
  charts.places.data.labels = pt.map(x => x[0]);
  charts.places.data.datasets[0].data = pt.map(x => x[1]);
  charts.places.update();
}

// ---------------------------------------------------------------- table
function cellVal(r, i){ return i < COLS.length ? r.cells[i] : r.st; }

function renderTable(scope, filteredCount){
  let rows = scope.slice();
  if (S.sortCol != null) {
    const i = S.sortCol, dir = S.sortDir;
    rows.sort((a, b) => {
      const x = cellVal(a, i), y = cellVal(b, i);
      const nx = Number(x), ny = Number(y);
      if (x !== "" && y !== "" && !isNaN(nx) && !isNaN(ny)) return (nx - ny) * dir;
      return String(x).localeCompare(String(y)) * dir;
    });
  }
  const size = +$("pageSize").value;
  const pages = Math.max(1, Math.ceil(rows.length / size));
  if (S.page >= pages) S.page = pages - 1;
  const start = S.page * size;
  const slice = rows.slice(start, start + size);

  $("thead").innerHTML = "<tr>" + COLS.concat(["map status (derived)"]).map((c, i) =>
    '<th data-i="'+i+'">'+esc(c)+(S.sortCol === i ? (S.sortDir === 1 ? " ▲" : " ▼") : "")+'</th>').join("") + "</tr>";

  $("tbody").innerHTML = slice.map(r => {
    const tds = r.cells.map((v, i) => {
      if (i === PRINTER_COL && v) return '<td title="'+esc(v)+'"><span class="plink" data-printer="'+esc(v)+'">'+esc(v)+'</span></td>';
      return '<td title="'+esc(v)+'">'+esc(v)+'</td>';
    }).join("");
    const st = r.st === "ok" ? "on map" : r.st;
    return '<tr data-i="'+r.i+'" class="'+(r.key && r.key === S.place ? "sel" : "")+'">'+tds+'<td>'+esc(st)+'</td></tr>';
  }).join("");

  $("tableInfo").textContent = rows.length
    ? "Showing " + (start + 1) + "–" + (start + slice.length) + " of " + rows.length + " records" +
      (rows.length !== R.length ? " (filtered from " + R.length + ")" : "")
    : "No records match the current filters.";
  $("pgInfo").textContent = " page " + (S.page + 1) + " / " + pages + " ";
  $("prevPg").disabled = S.page === 0;
  $("nextPg").disabled = S.page >= pages - 1;
}

$("thead").addEventListener("click", e => {
  const th = e.target.closest("th");
  if (!th) return;
  const i = +th.getAttribute("data-i");
  if (S.sortCol === i) S.sortDir = -S.sortDir; else { S.sortCol = i; S.sortDir = 1; }
  render();
});
$("tbody").addEventListener("click", e => {
  const link = e.target.closest(".plink");
  if (link) {
    const name = link.getAttribute("data-printer");
    $("printerInput").value = name;
    showTrail(name);
    return;
  }
  const tr = e.target.closest("tr");
  if (!tr) return;
  const r = R[+tr.getAttribute("data-i")];
  if (r && r.key) selectPlace(r.key, true);
});
$("pageSize").addEventListener("change", () => { S.page = 0; render(); });
$("prevPg").addEventListener("click", () => { S.page = Math.max(0, S.page - 1); render(); });
$("nextPg").addEventListener("click", () => { S.page += 1; render(); });
$("zoomBtn").addEventListener("click", () => fitToRows(S.scope));

// ---------------------------------------------------------------- Time Machine
let timer = null;
function stopPlay(){
  if (timer) { clearInterval(timer); timer = null; }
  $("playBtn").textContent = "▶ Play";
}
$("timeMode").addEventListener("change", () => {
  S.mode = $("timeMode").value;
  if (S.mode === "off") stopPlay();
  S.page = 0; render();
});
$("timeSlider").addEventListener("input", () => { S.t = +$("timeSlider").value; S.page = 0; render(); });
$("playBtn").addEventListener("click", () => {
  if (timer) { stopPlay(); return; }
  if (S.mode === "off") { S.mode = "cum"; $("timeMode").value = "cum"; }
  if (S.t >= D1) S.t = D0;
  $("timeSlider").value = S.t;
  $("playBtn").textContent = "⏸ Pause";
  render();
  timer = setInterval(() => {
    if (S.t >= D1) { stopPlay(); return; }
    S.t += 10;
    $("timeSlider").value = S.t;
    S.page = 0;
    render();
  }, 800);
});

// ---------------------------------------------------------------- Printer Trails
(function(){
  const byPrinter = {};
  R.forEach(r => {
    if (!r.printer || !r.key) return;
    (byPrinter[r.printer] = byPrinter[r.printer] || { keys:new Set(), n:0 });
    byPrinter[r.printer].keys.add(r.key);
    byPrinter[r.printer].n++;
  });
  const list = Object.entries(byPrinter).filter(([k, v]) => v.keys.size >= 2).sort((a, b) => b[1].n - a[1].n);
  $("printerList").innerHTML = list.map(([k]) => '<option value="'+esc(k)+'">').join("");
})();

function showTrail(name){
  trailLayer.clearLayers();
  const box = $("trailInfo");
  if (!name) { box.textContent = "Only printers recorded in two or more places are listed. You can also click a printer name in the table."; return; }
  const rows = R.filter(r => r.printer === name && r.lat != null && r.year != null).sort((a, b) => a.year - b.year);
  const stops = [];
  rows.forEach(r => {
    const last = stops[stops.length - 1];
    if (last && last.key === r.key) { last.y1 = r.year; last.n++; }
    else stops.push({ key:r.key, name:r.pi || r.place || "(no place name)", lat:r.lat, lon:r.lon, y0:r.year, y1:r.year, n:1 });
  });
  if (!stops.length) { box.textContent = "No dated editions with usable coordinates for this printer."; return; }
  if (stops.length >= 2) {
    L.polyline(stops.map(s => [s.lat, s.lon]), { color:"#1d4e89", weight:3, dashArray:"6 6", opacity:0.85 }).addTo(trailLayer);
  }
  stops.forEach((s, i) => {
    L.marker([s.lat, s.lon], { icon:L.divIcon({ className:"", html:'<div class="trail-num">'+(i+1)+'</div>', iconSize:[20,20], iconAnchor:[10,10] }), zIndexOffset:2000 })
      .bindTooltip((i+1)+". "+esc(s.name)+" ("+s.y0+(s.y1 !== s.y0 ? "–"+s.y1 : "")+")")
      .addTo(trailLayer);
  });
  box.innerHTML = "<b>"+esc(name)+"</b>"+(stops.length < 2 ? " – recorded at one place only." : "")+
    "<ol>"+stops.map(s => "<li>"+esc(s.name)+", "+s.y0+(s.y1 !== s.y0 ? "–"+s.y1 : "")+" ("+s.n+" edition"+(s.n === 1 ? "" : "s")+")</li>").join("")+"</ol>";
  map.fitBounds(L.latLngBounds(stops.map(s => [s.lat, s.lon])), { padding:[40,40], maxZoom:5 });
}
$("printerInput").addEventListener("change", () => showTrail($("printerInput").value.trim()));
$("clearTrail").addEventListener("click", () => { $("printerInput").value = ""; showTrail(""); });

// ---------------------------------------------------------------- data quality
(function(){
  const q = DATA.quality;
  let h = "<p>Coordinate parsing: " + Object.entries(q.status_counts).map(([k, v]) => esc(k)+" "+v).join(" · ") + "</p>";
  h += "<p><b>Places with more than one coordinate pair</b></p>";
  h += q.multi_coord_places.length
    ? "<ul>"+q.multi_coord_places.map(p => "<li>"+esc(p.place)+": "+p.pairs.map(x => x[0].toFixed(4)+", "+x[1].toFixed(4)).join(" | ")+"</li>").join("")+"</ul>"
    : "<p>None</p>";
  h += "<p><b>Coordinate pairs shared by more than one place</b></p>";
  h += q.shared_pairs.length
    ? "<ul>"+q.shared_pairs.map(p => "<li>"+p.lat.toFixed(4)+", "+p.lon.toFixed(4)+": "+p.places.map(esc).join(" | ")+"</li>").join("")+"</ul>"
    : "<p>None</p>";
  $("quality").innerHTML = h;
})();

// ---------------------------------------------------------------- main render
function syncControls(){
  const timeOn = S.mode !== "off";
  $("yMin").disabled = timeOn; $("yMax").disabled = timeOn;
  $("timeSlider").disabled = !timeOn;
  $("yLabel").textContent = S.yMin + "–" + S.yMax;
  $("timeLabel").textContent = S.mode === "cum" ? "Up to " + (S.t + 9)
                              : S.mode === "win" ? S.t + "s only" : "Time Machine is off";
  $("chips").innerHTML = S.place && S.groups.has(S.place)
    ? '<span class="chip" id="chipPlace" title="Click to remove">Place: '+esc(S.groups.get(S.place).name)+' ✕</span>' : "";
  const chip = $("chipPlace");
  if (chip) chip.addEventListener("click", () => { S.place = null; render(); });
}

function render(){
  const filtered = filterRows();
  S.groups = groupPlaces(filtered);
  if (S.place && !S.groups.has(S.place)) S.place = null;
  const scope = S.place ? filtered.filter(r => r.key === S.place) : filtered;
  S.filtered = filtered;
  S.scope = scope;
  syncControls();
  drawMap();
  updateLegend(filtered);
  renderKPIs(scope);
  renderCharts(scope);
  renderTable(scope, filtered.length);
}

render();
fitToRows(R);
})();
</script>
</body>
</html>
"""

html_text = HTML.replace("__DATA__", payload_json)
Path(OUTPUT_HTML).write_text(html_text, encoding="utf-8")

usable = sum(1 for r in records if r["st"] == "ok")
print(f"Records written to page: {len(records)} (usable coordinates: {usable}, "
      f"{usable / len(records) * 100:.1f}%)")
print(f"Created {OUTPUT_HTML}. Open it in a web browser (an internet connection is needed for the map tiles and libraries).")

# In Jupyter, show a clickable download link for the finished file.
# (Outside Jupyter this part is skipped quietly.)
try:
    from IPython.display import FileLink, display
    display(FileLink(OUTPUT_HTML, result_html_prefix="Download: "))
except Exception:
    pass

FileNotFoundError: [Errno 2] No such file or directory: '-f'

In [12]:
!python build_printing_places_site.py

Loaded 500 rows and 19 columns from book-places-wikidata.csv
edition_id is unique: 500 values
Usable coordinates: 499 of 500 (99.8%)
Created printing_places_dashboard.html. Open it in a web browser (an internet connection is needed for the map tiles and libraries).
C:\Users\Pavlo\printing_places_dashboard.html
